<a href="https://colab.research.google.com/github/KKTRKKT/AI_study/blob/main/ai-%EC%B1%8C%EB%A6%B0%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# GPU 확인 — H100이 배정됐는지 먼저 확인!
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU 정보:', result.stdout.strip())

# H100이 아닌 경우 배치/lr 조정 안내
gpu_name = result.stdout.strip()
if 'H100' in gpu_name:
    print('H100 확인! Full FT 설정으로 진행합니다.')
elif 'A100' in gpu_name:
    print('A100 감지. BATCH_SIZE=4, GRAD_ACCUM=4 로 변경을 권장합니다.')
else:
    print('주의: H100/A100이 아닙니다. VRAM 부족 가능성이 있습니다.')

GPU 정보: NVIDIA H100 80GB HBM3, 81559 MiB
H100 확인! Full FT 설정으로 진행합니다.


In [ ]:
# Colab은 기본 torch가 설치되어 있으므로 핵심 라이브러리만 설치
# ✅ [변경] bitsandbytes, peft 제거 — Full FT에서는 불필요
# ✅ [추가] flash-attn — H100에서 학습 속도 2배 향상
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q "accelerate>=0.34.2" datasets pillow pandas qwen-vl-utils --upgrade
!pip install -q flash-attn --no-build-isolation

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 148.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 164.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 81.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pillow<1

In [ ]:
import torch
print('Torch 버전:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from collections import Counter

# ✅ [변경] BitsAndBytesConfig, peft 관련 import 전부 제거
# 이유: Full FT는 양자화 없이 전체 가중치를 학습하므로 불필요
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    get_cosine_schedule_with_warmup,  # ✅ [변경] linear → cosine (Full FT에 더 안정적)
)
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# ============================
# 실험 설정
# ============================
EXP_NUM    = 1           # ← 새 실험마다 여기만 변경!
MODEL_ID   = 'Qwen/Qwen3-VL-8B-Instruct'
IMAGE_SIZE = 512
MAX_NEW_TOKENS = 8
SEED       = 42

# ✅ [변경] H100 Full FT 최적 하이퍼파라미터
BATCH_SIZE = 16          # H100 80GB 기준. A100이면 4로 변경
GRAD_ACCUM = 1           # 배치가 크므로 누적 불필요. A100이면 4로 변경
NUM_EPOCHS = 3           # Full FT는 epoch 늘려야 수렴
LR         = 1e-5        # ✅ [변경] 4e-4 → 1e-5 (Full FT는 낮게 설정해야 발산 방지)
WARMUP_RATIO = 0.05      # 전체 스텝의 5%를 warmup에 사용

# 기본 경로
BASE_DIR = '/content/2026-ssafy-15-2-ai'

# 저장 경로
SAVE_DIR = f'{BASE_DIR}/qwen3_fullft_exp{EXP_NUM}_best'
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ✅ [추가] H100 전용 TF32 활성화 — 속도 향상, 정확도 유지
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

train_df = pd.read_csv(f'{BASE_DIR}/train.csv')
test_df  = pd.read_csv(f'{BASE_DIR}/test.csv')

# answer 결측치 제거 (EDA에서 1개 확인됨)
train_df = train_df.dropna(subset=['answer']).reset_index(drop=True)

print(f'학습 데이터: {len(train_df)}개')
print(f'테스트 데이터: {len(test_df)}개')
print(f'저장 경로: {SAVE_DIR}')

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# ✅ [변경] 4bit 양자화 완전 제거 → bfloat16 Full precision 로드
# ✅ [추가] attn_implementation='flash_attention_2' → H100 NVLink 최대 활용
# ✅ [변경] device_map='cuda:0' → 'auto' (H100 80GB면 단일 GPU에 전부 올라감)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
    device_map='auto',
    trust_remote_code=True,
)

# ✅ [제거] prepare_model_for_kbit_training — kbit 양자화 없으므로 불필요
# ✅ [추가] gradient_checkpointing — Full FT에서도 VRAM 안전 확보용
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

# ✅ [제거] LoraConfig, get_peft_model — Full FT에서 불필요

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total_params/1e9:.2f}B')
print(f'학습 가능 파라미터: {trainable_params/1e9:.2f}B ({100*trainable_params/total_params:.1f}%)')
print(f'실험 저장 경로 (로컬): {LOCAL_SAVE_DIR}')
print(f'실험 저장 경로 (Drive): {DRIVE_SAVE_DIR}')

In [ ]:
SYSTEM_INSTRUCT = (
    "당신은 재활용품 분류를 위한 시각적 질의응답 어시스턴트입니다. "
    "답변하기 전에 반드시 이미지를 주의 깊게 살펴보아야 합니다. "
    "텍스트의 길이나 단어 패턴을 보고 추측하지 마세요. "
    "이미지에 실제로 보이는 물체, 재질, 라벨에만 집중하십시오. "
    "설명 없이 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 답변하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f'{question}\n'
        f'(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n'
        '정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.'
    )

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        q = str(row['question'])
        a, b, c, d = str(row['a']), str(row['b']), str(row['c']), str(row['d'])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user',   'content': [
                {'type': 'image', 'image': img},
                {'type': 'text',  'text': user_text}
            ]}
        ]
        if self.train:
            gold = str(row['answer']).strip().lower()
            messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': gold}]})

        return {'messages': messages, 'image': img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample['messages'], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample['image'])

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors='pt'
        )

        if self.train:
            # ✅ [변경] 전체 시퀀스 loss → assistant 답변 토큰만 loss 계산
            # 이유: 시스템 프롬프트/질문 부분은 모델이 '생성'하는 게 아님
            # 정답(a/b/c/d) 토큰에만 집중해야 학습 효율 UP
            labels = enc['input_ids'].clone()

            # assistant 토큰 이전 구간은 -100으로 마스킹 (loss 계산 제외)
            # Qwen3 채팅 템플릿의 assistant 시작 마커: '<|im_start|>assistant\n'
            assistant_token_ids = self.processor.tokenizer.encode(
                '<|im_start|>assistant\n', add_special_tokens=False
            )
            for i, label_row in enumerate(labels):
                # assistant 마커 위치 탐색
                mask_until = len(label_row)  # 기본값: 전부 마스킹
                for j in range(len(label_row) - len(assistant_token_ids)):
                    if label_row[j:j+len(assistant_token_ids)].tolist() == assistant_token_ids:
                        mask_until = j + len(assistant_token_ids)
                        break
                labels[i, :mask_until] = -100

            # padding 토큰도 마스킹
            labels[enc['attention_mask'] == 0] = -100
            enc['labels'] = labels

        return enc

In [ ]:
train_df_shuffled = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split = int(len(train_df_shuffled) * 0.9)
train_subset = train_df_shuffled.iloc[:split].reset_index(drop=True)
valid_subset = train_df_shuffled.iloc[split:].reset_index(drop=True)

print(f'Train: {len(train_subset)}개, Valid: {len(valid_subset)}개')

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# ✅ [변경] batch_size 1 → BATCH_SIZE (16), num_workers 0 → 4
# 이유: H100은 VRAM 여유가 많아 큰 배치 사용 가능
# num_workers=4로 데이터 로딩 병렬화 → GPU idle 시간 감소
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=4, pin_memory=True
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=4, pin_memory=True
)

In [ ]:
from tqdm.auto import tqdm

def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return 'a'
    last = lines[-1]
    if last in ['a', 'b', 'c', 'd']:
        return last
    for line in reversed(lines):
        for tok in line.split():
            tok_clean = tok.strip('.,!?()[]')
            if tok_clean in ['a', 'b', 'c', 'd']:
                return tok_clean
    matches = re.findall(r'\b([abcd])\b', text)
    if matches:
        return matches[-1]
    counts = Counter(re.findall(r'[abcd]', text))
    if counts:
        return counts.most_common(1)[0][0]
    return 'a'


model = model.to(device)
AMP_DTYPE = torch.bfloat16

# ✅ [변경] lr 4e-4 → 1e-5 (Full FT는 낮은 lr 필수)
# 이유: 전체 가중치를 업데이트하므로 큰 lr은 사전학습 지식을 파괴함
# ✅ [추가] weight_decay=0.01 (L2 정규화로 과적합 방지)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)

# ✅ [변경] linear → cosine 스케줄러 (Full FT에서 더 안정적인 수렴)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * WARMUP_RATIO),
    num_training_steps=num_training_steps
)

# ✅ [변경] GradScaler 완전 제거 — bfloat16 + Full FT에서 불필요
# 이유: bf16은 fp16과 달리 GradScaler 없이도 학습 안정적

best_val_acc = 0.0
global_step  = 0

print(f'총 학습 스텝: {num_training_steps}')
print(f'Warmup 스텝: {int(num_training_steps * WARMUP_RATIO)}')
print(f'배치 크기: {BATCH_SIZE}, Grad Accum: {GRAD_ACCUM}, 유효 배치: {BATCH_SIZE * GRAD_ACCUM}')
print('─' * 50)

for epoch in range(NUM_EPOCHS):
    # ───────────────── Train ─────────────────
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]', unit='batch')

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        # ✅ [변경] scaler 분기 제거 → 직접 backward
        loss.backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            # ✅ [추가] Gradient Clipping — Full FT에서 gradient 폭발 방지
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            current_lr = scheduler.get_last_lr()[0]
            progress_bar.set_postfix({
                'loss': f'{running_loss:.3f}',
                'lr': f'{current_lr:.2e}'
            })
            running_loss = 0.0

    # ───────────────── Validation ─────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for row in tqdm(valid_subset.itertuples(), total=len(valid_subset),
                        desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [valid]'):
            img = Image.open(row.path).convert('RGB')
            user_text = build_mc_prompt(row.question, row.a, row.b, row.c, row.d)
            messages = [
                {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
                {'role': 'user',   'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': user_text}]}
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[img], return_tensors='pt').to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out_ids = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    eos_token_id=processor.tokenizer.eos_token_id
                )
            pred = extract_choice(processor.batch_decode(out_ids, skip_special_tokens=True)[0])
            correct += int(pred == str(row.answer).strip().lower())

            # VRAM 해제
            del inputs
            torch.cuda.empty_cache()

    val_acc = correct / len(valid_subset)
    print(f'[Epoch {epoch+1}] Valid Acc: {val_acc:.4f} ({correct}/{len(valid_subset)})')

    # ✅ [변경] LoRA adapter 저장 → 전체 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f'  ★ Best model saved (acc={best_val_acc:.4f}) → {SAVE_DIR}')

    model.train()

print(f'\n학습 완료. Best Valid Acc: {best_val_acc:.4f}')

In [ ]:
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc='Inference', unit='sample'):
    row = test_df.iloc[i]
    try:
        img = Image.open(row['path']).convert('RGB')
        user_text = build_mc_prompt(row['question'], row['a'], row['b'], row['c'], row['d'])

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user',   'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': user_text}]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors='pt').to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id
            )
        output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
        preds.append(extract_choice(output_text))

    except Exception as e:
        print(f'[Sample {i}] 에러: {e}')
        preds.append('a')

    finally:
        del inputs
        torch.cuda.empty_cache()

# 제출 파일 생성
submission = pd.DataFrame({'id': test_df['id'], 'answer': preds})
submission.to_csv(f'{BASE_DIR}/submission.csv', index=False)

print('Saved submission.csv')
print(f'경로: {BASE_DIR}/submission.csv')
print(f'Answer distribution:\n{submission["answer"].value_counts()}')

In [ ]:
# 모델 응답 예시 확인
print(output_text)